In [1]:
from datetime import UTC, datetime

import pandas as pd
import pyvo

TAP_ENDPOINT = "https://almascience.eso.org/tap"
CAPTURED_AT = datetime.now(UTC).isoformat()

service = pyvo.dal.TAPService(TAP_ENDPOINT)

print("Endpoint:", TAP_ENDPOINT)
print("Captured at:", CAPTURED_AT)

Endpoint: https://almascience.eso.org/tap
Captured at: 2026-08-31T12:27:55.081125+00:00


In [2]:
FIELDS = (
    "s_resolution",
    "spatial_resolution",
    "frequency",
    "bandwidth",
    "frequency_support",
    "spectral_resolution",
    "em_resolution",
    "sensitivity_10kms",
    "cont_sensitivity_bandwidth",
    "type",
    "qa2_passed",
)

field_literals = ", ".join(
    f"'{field_name}'"
    for field_name in FIELDS
)

metadata_adql = f"""
SELECT
    column_name,
    datatype,
    arraysize,
    unit,
    ucd,
    description
FROM TAP_SCHEMA.columns
WHERE table_name = 'ivoa.obscore'
  AND column_name IN ({field_literals})
ORDER BY column_name
"""

metadata = (
    service.search(
        metadata_adql,
        maxrec=100,
    )
    .to_table()
    .to_pandas()
)

metadata

,column_name,datatype,arraysize,unit,ucd,description
0,bandwidth,double,,Hz,em.freq;instr.bandpass,Total Bandwidth
1,cont_sensitivity_bandwidth,double,,mJy/beam,,Estimated noise in the aggregated continuum ba...
2,em_resolution,double,,m,spect.resolution;stat.mean,Estimated frequency resolution from all the sp...
3,frequency,double,,GHz,em.freq;obs;meta.main,Observed (tuned) reference frequency on the sky.
4,frequency_support,char,4000*,GHz,em.freq;obs;meta.main,All frequency ranges used by the field
5,qa2_passed,char,,,meta.code,Quality Assessment 2 status: does the Member /...
6,s_resolution,double,,arcsec,pos.angResolution,typical spatial resolution
7,sensitivity_10kms,double,,mJy/beam,,Estimated noise in an nominal 10km/s bandwidth...
8,spatial_resolution,double,,arcsec,,Average of the maximum and minimum spatial res...
9,spectral_resolution,double,,kHz,,


In [3]:
assert set(metadata["column_name"]) == set(FIELDS)
assert not metadata["column_name"].duplicated().any()

metadata[
    [
        "column_name",
        "datatype",
        "arraysize",
        "unit",
        "ucd",
        "description",
    ]
]

,column_name,datatype,arraysize,unit,ucd,description
0,bandwidth,double,,Hz,em.freq;instr.bandpass,Total Bandwidth
1,cont_sensitivity_bandwidth,double,,mJy/beam,,Estimated noise in the aggregated continuum ba...
2,em_resolution,double,,m,spect.resolution;stat.mean,Estimated frequency resolution from all the sp...
3,frequency,double,,GHz,em.freq;obs;meta.main,Observed (tuned) reference frequency on the sky.
4,frequency_support,char,4000*,GHz,em.freq;obs;meta.main,All frequency ranges used by the field
5,qa2_passed,char,,,meta.code,Quality Assessment 2 status: does the Member /...
6,s_resolution,double,,arcsec,pos.angResolution,typical spatial resolution
7,sensitivity_10kms,double,,mJy/beam,,Estimated noise in an nominal 10km/s bandwidth...
8,spatial_resolution,double,,arcsec,,Average of the maximum and minimum spatial res...
9,spectral_resolution,double,,kHz,,


In [4]:
type_count_adql = """
SELECT
    type,
    COUNT(*) AS row_count
FROM ivoa.obscore
WHERE science_observation = 'T'
GROUP BY type
"""

type_counts = (
    service.search(
        type_count_adql,
        maxrec=1000,
    )
    .to_table()
    .to_pandas()
    .sort_values(
        "row_count",
        ascending=False,
    )
    .reset_index(drop=True)
)

type_counts

,type,row_count
0,S,356866
1,L,81677
2,T,3815
3,V,600
4,SV,145
5,E,64
6,P,40
7,CAL,4


In [5]:
type_sample_adql = """
SELECT TOP 200
    proposal_id,
    member_ous_uid,
    obs_id,
    type,
    frequency,
    bandwidth,
    frequency_support,
    spectral_resolution,
    em_resolution,
    pol_states
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND type IS NOT NULL
"""

type_sample = (
    service.search(
        type_sample_adql,
        maxrec=200,
    )
    .to_table()
    .to_pandas()
)

type_sample.head(20)

,proposal_id,member_ous_uid,obs_id,type,frequency,bandwidth,frequency_support,spectral_resolution,em_resolution,pol_states
0,2021.1.00869.L,uid://A001/X1590/X18e0,uid://A001/X1590/X18e0.source.ad3a-22078.spw.31,L,87.492595,9.375000e+08,"[84.10..85.04GHz,294.68kHz,3.8mJy/beam@10km/s,...",294.677734,1.154085e-08,/XX/YY/
1,2021.1.00869.L,uid://A001/X1590/X18e0,uid://A001/X1590/X18e0.source.ad3a-22078.spw.25,L,84.566616,9.375000e+08,"[84.10..85.04GHz,294.68kHz,3.8mJy/beam@10km/s,...",294.677734,1.235331e-08,/XX/YY/
2,2021.1.00869.L,uid://A001/X1590/X18e0,uid://A001/X1590/X18e0.source.ad3a-22078.spw.27,L,85.398372,9.375000e+08,"[84.10..85.04GHz,294.68kHz,3.8mJy/beam@10km/s,...",294.677734,1.211384e-08,/XX/YY/
3,2021.1.00869.L,uid://A001/X1590/X18e0,uid://A001/X1590/X18e0.source.ad3a-22078.spw.29,L,86.545116,9.375000e+08,"[84.10..85.04GHz,294.68kHz,3.8mJy/beam@10km/s,...",294.677734,1.179494e-08,/XX/YY/
4,2024.1.01553.S,uid://A001/X3788/Xb661,uid://A001/X3788/Xb661.source.3C078.spw.19,S,225.732519,2.000000e+09,"[222.90..224.77GHz,7812.01kHz,838.3uJy/beam@10...",31250.000000,1.838615e-07,/XX/YY/
5,2024.1.01553.S,uid://A001/X3788/Xb661,uid://A001/X3788/Xb661.source.3C078.spw.21,S,237.290863,2.000000e+09,"[222.90..224.77GHz,7812.01kHz,838.3uJy/beam@10...",31250.000000,1.663858e-07,/XX/YY/
6,2024.1.01553.S,uid://A001/X3788/Xb661,uid://A001/X3788/Xb661.source.3C078.spw.23,S,240.296886,2.000000e+09,"[222.90..224.77GHz,7812.01kHz,838.3uJy/beam@10...",31250.000000,1.622489e-07,/XX/YY/
7,2024.1.01553.S,uid://A001/X3788/Xb661,uid://A001/X3788/Xb661.source.3C078.spw.25,S,223.836776,1.875000e+09,"[222.90..224.77GHz,7812.01kHz,838.3uJy/beam@10...",7812.011719,4.674426e-08,/XX/YY/
8,2021.1.00869.L,uid://A001/X1590/X18dc,uid://A001/X1590/X18dc.source.ad3a-21468.spw.25,L,84.567160,9.375000e+08,"[84.10..85.04GHz,294.68kHz,4.5mJy/beam@10km/s,...",294.677734,1.235315e-08,/XX/YY/
9,2021.1.00869.L,uid://A001/X1590/X18dc,uid://A001/X1590/X18dc.source.ad3a-21468.spw.27,L,85.398893,9.375000e+08,"[84.10..85.04GHz,294.68kHz,4.5mJy/beam@10km/s,...",294.677734,1.211369e-08,/XX/YY/


In [6]:
search_text = (
    type_sample[
        [
            "type",
            "frequency_support",
        ]
    ]
    .astype(str)
    .agg(" ".join, axis=1)
    .str.lower()
)

for token in (
    "fdm",
    "tdm",
    "continuum",
    "line",
):
    print(
        token,
        int(search_text.str.contains(token).sum()),
    )

fdm 0
tdm 0
continuum 0
line 0


In [7]:
with pd.option_context(
    "display.max_colwidth",
    None,
    "display.max_rows",
    None,
):
    display(
        metadata[
            [
                "column_name",
                "datatype",
                "arraysize",
                "unit",
                "ucd",
                "description",
            ]
        ]
    )

type_census_total = int(
    type_counts["row_count"].sum()
)

count_adql = """
SELECT COUNT(*) AS total_matches
FROM ivoa.obscore
WHERE science_observation = 'T'
"""

count_result = (
    service.search(
        count_adql,
        maxrec=1,
    )
    .to_table()
    .to_pandas()
)

server_total = int(
    count_result.iloc[0, 0]
)

print(
    "Sum of grouped type counts:",
    type_census_total,
)
print(
    "Direct server COUNT:",
    server_total,
)

assert type_census_total == server_total

,column_name,datatype,arraysize,unit,ucd,description
0,bandwidth,double,,Hz,em.freq;instr.bandpass,Total Bandwidth
1,cont_sensitivity_bandwidth,double,,mJy/beam,,"Estimated noise in the aggregated continuum bandwidth. Note this is an indication only, it does not include the effects of flagging or dynamic range limitations."
2,em_resolution,double,,m,spect.resolution;stat.mean,"Estimated frequency resolution from all the spectral windows, using median values of channel widths."
3,frequency,double,,GHz,em.freq;obs;meta.main,Observed (tuned) reference frequency on the sky.
4,frequency_support,char,4000*,GHz,em.freq;obs;meta.main,All frequency ranges used by the field
5,qa2_passed,char,,,meta.code,Quality Assessment 2 status: does the Member / Group OUS fulfil the PI's requirements?
6,s_resolution,double,,arcsec,pos.angResolution,typical spatial resolution
7,sensitivity_10kms,double,,mJy/beam,,"Estimated noise in an nominal 10km/s bandwidth. Note this is an indication only, it does not include the effects of flagging or Hanning smoothing, and a 10km/s bandwidth may not be achievable with the data as taken."
8,spatial_resolution,double,,arcsec,,Average of the maximum and minimum spatial resolution values of all spectral windows
9,spectral_resolution,double,,kHz,,


Sum of grouped type counts: 443211
Direct server COUNT: 443211


In [8]:
type_example_frames = []

for type_code in (
    type_counts["type"]
    .astype(str)
    .tolist()
):
    example_adql = f"""
    SELECT TOP 3
        proposal_id,
        member_ous_uid,
        obs_id,
        type,
        frequency_support
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND type = '{type_code}'
    """

    example_frame = (
        service.search(
            example_adql,
            maxrec=3,
        )
        .to_table()
        .to_pandas()
    )

    type_example_frames.append(
        example_frame
    )

type_examples = pd.concat(
    type_example_frames,
    ignore_index=True,
)

display(
    type_examples[
        [
            "proposal_id",
            "type",
            "obs_id",
            "frequency_support",
        ]
    ]
)

print(
    type_examples[
        "type"
    ].value_counts()
)

,proposal_id,type,obs_id,frequency_support
0,2024.1.01553.S,S,uid://A001/X3788/Xb661.source.3C078.spw.19,"[222.90..224.77GHz,7812.01kHz,838.3uJy/beam@10..."
1,2024.1.01553.S,S,uid://A001/X3788/Xb661.source.3C078.spw.21,"[222.90..224.77GHz,7812.01kHz,838.3uJy/beam@10..."
2,2024.1.01553.S,S,uid://A001/X3788/Xb661.source.3C078.spw.23,"[222.90..224.77GHz,7812.01kHz,838.3uJy/beam@10..."
3,2019.1.01634.L,L,uid://A001/X14d7/X1ae.source.XMM1-Z-151269.spw.27,"[234.90..236.77GHz,7812.01kHz,793.7uJy/beam@10..."
4,2019.1.01634.L,L,uid://A001/X14d7/X1ae.source.XMM1-Z-151269.spw.29,"[234.90..236.77GHz,7812.01kHz,793.7uJy/beam@10..."
5,2019.1.01634.L,L,uid://A001/X14d7/X1ae.source.XMM1-Z-151269.spw.31,"[234.90..236.77GHz,7812.01kHz,793.7uJy/beam@10..."
6,2017.A.00046.T,T,uid://A001/X1306/X3b.source.AT2018cowATLAS18qq...,"[89.50..91.49GHz,31250.00kHz,5.3mJy/beam@10km/..."
7,2017.A.00046.T,T,uid://A001/X1306/X3b.source.AT2018cowATLAS18qq...,"[89.50..91.49GHz,31250.00kHz,5.3mJy/beam@10km/..."
8,2017.A.00046.T,T,uid://A001/X1306/X3b.source.AT2018cowATLAS18qq...,"[89.50..91.49GHz,31250.00kHz,5.3mJy/beam@10km/..."
9,2017.1.00841.V,V,uid://A001/X12d1/X25.source.m87.spw.10,"[212.17..214.04GHz,7808.59kHz,754.7uJy/beam@10..."


type
S      3
L      3
T      3
V      3
SV     3
E      3
P      3
CAL    3
Name: count, dtype: int64


In [9]:
stratified_search_text = (
    type_examples[
        [
            "type",
            "frequency_support",
        ]
    ]
    .astype(str)
    .agg(" ".join, axis=1)
    .str.lower()
)

stratified_token_counts = {}

for token in (
    "fdm",
    "tdm",
    "continuum",
    "line",
):
    token_count = int(
        stratified_search_text
        .str.contains(
            token,
            regex=False,
        )
        .sum()
    )

    stratified_token_counts[token] = (
        token_count
    )

    print(
        token,
        token_count,
    )

fdm 0
tdm 0
continuum 0
line 0


In [10]:
project_type_adql = """
SELECT DISTINCT
    proposal_id,
    type
FROM ivoa.obscore
WHERE science_observation = 'T'
"""

project_types = (
    service.search(
        project_type_adql,
        maxrec=20000,
    )
    .to_table()
    .to_pandas()
)

project_types[
    "proposal_suffix"
] = (
    project_types[
        "proposal_id"
    ]
    .astype(str)
    .str.rsplit(
        ".",
        n=1,
    )
    .str[-1]
)

type_suffix_table = pd.crosstab(
    project_types["type"],
    project_types["proposal_suffix"],
)

display(type_suffix_table)

type_suffix_mismatches = (
    project_types[
        project_types["type"]
        != project_types["proposal_suffix"]
    ]
    .sort_values(
        [
            "type",
            "proposal_id",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Distinct project/type pairs:",
    len(project_types),
)
print(
    "Type/suffix mismatches:",
    len(type_suffix_mismatches),
)

display(
    type_suffix_mismatches.head(30)
)

proposal_suffix,CAL,E,L,P,S,SV,T,V
type,,,,,,,,
CAL,1,0,0,0,0,0,0,0
E,0,6,0,0,0,0,0,0
L,0,0,38,0,0,0,0,0
P,0,0,0,3,0,0,0,0
S,0,0,0,0,5348,0,0,0
SV,0,0,0,0,0,17,0,0
T,0,0,0,0,0,0,136,0
V,0,0,0,0,0,0,0,65


Distinct project/type pairs: 5614
Type/suffix mismatches: 0


,proposal_id,type,proposal_suffix


In [11]:
assert type_suffix_mismatches.empty

In [12]:
support_token_variants = {
    "continuum": (
        "continuum",
        "Continuum",
        "CONTINUUM",
    ),
    "line": (
        "line",
        "Line",
        "LINE",
    ),
    "fdm": (
        "fdm",
        "Fdm",
        "FDM",
    ),
    "tdm": (
        "tdm",
        "Tdm",
        "TDM",
    ),
}

support_token_results = []

for token_name, variants in (
    support_token_variants.items()
):
    like_clauses = " OR ".join(
        "frequency_support "
        f"LIKE '%{variant}%'"
        for variant in variants
    )

    token_count_adql = f"""
    SELECT COUNT(*) AS matching_rows
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND (
          {like_clauses}
      )
    """

    token_count_result = (
        service.search(
            token_count_adql,
            maxrec=1,
        )
        .to_table()
        .to_pandas()
    )

    support_token_results.append(
        {
            "token": token_name,
            "matching_rows": int(
                token_count_result.iloc[
                    0,
                    0,
                ]
            ),
        }
    )

support_token_census = pd.DataFrame(
    support_token_results
)

display(support_token_census)

,token,matching_rows
0,continuum,0
1,line,0
2,fdm,0
3,tdm,0


In [13]:
all_column_metadata_adql = """
SELECT
    column_name,
    datatype,
    arraysize,
    unit,
    ucd,
    description
FROM TAP_SCHEMA.columns
WHERE table_name = 'ivoa.obscore'
ORDER BY column_name
"""

all_column_metadata = (
    service.search(
        all_column_metadata_adql,
        maxrec=200,
    )
    .to_table()
    .to_pandas()
)

EXPECTED_COLUMN_COUNT_AT_CAPTURE = 73

current_column_count = len(
    all_column_metadata
)

print(
    "Current ivoa.obscore column count:",
    current_column_count,
)

if (
    current_column_count
    != EXPECTED_COLUMN_COUNT_AT_CAPTURE
):
    print(
        "Schema count changed since the "
        "2026-08-31 capture:",
        EXPECTED_COLUMN_COUNT_AT_CAPTURE,
        "->",
        current_column_count,
    )

assert set(FIELDS).issubset(
    set(
        all_column_metadata[
            "column_name"
        ]
    )
)

semantic_search_text = (
    all_column_metadata["column_name"]
    .fillna("")
    .astype(str)
    + " "
    + all_column_metadata["description"]
    .fillna("")
    .astype(str)
)

semantic_column_mask = (
    semantic_search_text.str.contains(
        (
            "type|mode|correlator|"
            "continuum|line|tdm|fdm|"
            "frequency|spectral|"
            "spectral window|spw|"
            "resolution|sensitivity"
        ),
        case=False,
        regex=True,
    )
)

semantic_columns = (
    all_column_metadata[
        semantic_column_mask
    ]
    .reset_index(drop=True)
)

with pd.option_context(
    "display.max_colwidth",
    None,
    "display.max_rows",
    None,
):
    display(
        semantic_columns[
            [
                "column_name",
                "datatype",
                "unit",
                "ucd",
                "description",
            ]
        ]
    )

Current ivoa.obscore column count: 73


,column_name,datatype,unit,ucd,description
0,calib_level,int,,meta.code;obs.calib,"calibration level (2 or 3). 2 if product_type = MOUS, 3 if product_type = GOUS"
1,cont_sensitivity_bandwidth,double,mJy/beam,,"Estimated noise in the aggregated continuum bandwidth. Note this is an indication only, it does not include the effects of flagging or dynamic range limitations."
2,dataproduct_type,char,,meta.code.class,type of product
3,em_max,double,m,em.wl;stat.max,stop spectral coordinate value
4,em_min,double,m,em.wl;stat.min,start spectral coordinate value
5,em_res_power,double,,spect.resolution,typical spectral resolution
6,em_resolution,double,m,spect.resolution;stat.mean,"Estimated frequency resolution from all the spectral windows, using median values of channel widths."
7,em_xel,int,,meta.number,Number of elements along the spectral axis
8,frequency,double,GHz,em.freq;obs;meta.main,Observed (tuned) reference frequency on the sky.
9,frequency_support,char,GHz,em.freq;obs;meta.main,All frequency ranges used by the field


## Semantic closure decisions

Capture timestamp: `2026-08-31T12:27:55.081125+00:00`

### Service metadata

* All eleven selected production and semantic-review fields were present in
  `TAP_SCHEMA.columns`.
* `frequency` is declared in GHz.
* `bandwidth` is declared in Hz.
* `spectral_resolution` is declared in kHz.
* `s_resolution` and `spatial_resolution` are both declared in arcsec, but
  their service descriptions differ and they remain separate evidence.
* `sensitivity_10kms` and `cont_sensitivity_bandwidth` are both declared in
  mJy/beam but represent different sensitivity bases.
* `frequency_support` has a declared top-level unit of GHz while its raw
  composite representation contains quantities with multiple units.

### Snapshot evidence

* The current `ivoa.obscore` schema contained 73 columns.
* The complete science-target census contained 443,211 rows.
* The sum of the grouped `type` counts exactly matched the direct
  server-side `COUNT(*)`.
* Across all 443,211 science-target rows, the standard literal forms of
  `continuum`, `line`, `FDM`, and `TDM` each occurred zero times in the
  `frequency_support` raw string.
* No separate correlator-mode column was identified in the current
  `ivoa.obscore` schema.
* These results describe the captured Archive state and must not be treated
  as immutable properties of a continuously changing external service.

### Archive type field

The complete current science-target census observed the following `type`
values:

`S`, `L`, `T`, `V`, `SV`, `E`, `P`, and `CAL`.

These values represent proposal/project classes encoded by the terminal
`proposal_id` suffix. Across 5,614 distinct current proposal/type pairs,
no mismatch between `type` and the terminal proposal suffix was observed.

The observed values must not be treated as a permanently closed enumeration.
A later production model should preserve unknown or newly introduced values
explicitly.

The top-level `type` field does not represent science intent,
spectral-window type, or correlator mode. No FDM/TDM classification is
derived from this field.

The value `type = 'T'` is a proposal class, while
`science_observation = 'T'` is a boolean-like science-row flag. They are
unrelated fields.

### Frequency-Support mode representation

The Cycle 13 ALMA Science Archive Manual documents a nested Frequency
Support `Type` value:

* `continuum` maps to TDM;
* `line` maps to FDM.

This documented nested Type is distinct from the top-level TAP `type`
column, which represents proposal/project classification.

In the current `ivoa.obscore` TAP snapshot, the documented nested Frequency
Support Type was not observed as literal text in the `frequency_support`
raw representation and was not exposed as a separate `ivoa.obscore`
column.

The Archive-wide token census searched the current science-target population
for the standard literal forms of `continuum`, `line`, `FDM`, and `TDM`.
All four searches returned zero matching rows.

This result does not prove that the documented value is absent from every
ALMA service representation. It establishes only that the value was not
visible in the tested current `ivoa.obscore` raw strings or columns.

The unresolved question is therefore not the scientific meaning of
`continuum` and `line`. The unresolved question is how, or whether, the
Archive Query Interface representation can be obtained through a stable
production API.

### Representation gap

Official Archive documentation defines the semantic relationship between the
nested Frequency Support Type and correlator mode. However, the current
`ivoa.obscore` TAP representation does not expose that value in the tested
raw string or as a separate ObsCore column.

A future investigation should determine whether the value is available
through:

1. another documented TAP table;
2. a stable Archive API;
3. DataLink or another VO representation;
4. an Archive Query Interface-specific response.

No production dependency on an undocumented web-interface endpoint should
be introduced.

### Sensitivity evidence basis

Both Archive sensitivity fields are estimated metadata rather than achieved
image-product RMS measurements.

* `cont_sensitivity_bandwidth` is the estimated noise over the aggregated
  continuum bandwidth.
* `sensitivity_10kms` is the estimated line sensitivity at a nominal
  10 km/s bandwidth.
* The service metadata states that these values do not fully include effects
  such as flagging, dynamic-range limitations, or Hanning smoothing.
* A nominal 10 km/s bandwidth may not be achievable for every dataset.

These fields may be used as Archive candidate evidence, but they must not be
labelled as achieved QA2 product sensitivity.

The line and continuum sensitivity fields must remain semantically separate.
Their values must not be substituted for one another without an explicit
duplication-policy rule.

### Angular-resolution evidence

`spatial_resolution` remains the primary Archive angular-resolution field
for candidate retrieval and comparison because it is the field used by the
official ALMA spatial-resolution query examples.

`s_resolution` remains separate ObsCore evidence. It must not be silently
substituted for `spatial_resolution`, because the service descriptions and
observed values are not identical.

Neither field should be represented as a measured FITS restoring beam.
Product-level achieved beam measurements remain outside the current Archive
ingestion scope.

### QA2 boundary

`qa2_passed` is retained as quality-state evidence. It is not used as a
default retrieval filter.

Archive observational metadata may become available after QA0 while later
processing or QA2 assessment is still incomplete. Excluding all rows without
`qa2_passed = 'T'` would therefore introduce a policy decision into the
ingestion layer and could hide relevant duplication candidates.

Any future decision to include or exclude observations according to QA2
state must be implemented as an explicit policy-layer rule rather than an
implicit Archive-client restriction.

### Engineering decisions

* Do not modify the current `frequency_support` parser.
* Do not add top-level `type` to the v1 production retrieval projection.
* Do not derive FDM/TDM from the top-level TAP `type` column.
* Do not infer FDM/TDM solely from bandwidth or spectral resolution.
* Treat the documented nested Frequency Support Type as unavailable in the
  current `ivoa.obscore` TAP representation until a stable source
  representation is identified.
* Do not fabricate a mode field that is absent from the raw TAP value.
* Preserve `type` as proposal/project classification metadata when it is
  ingested by a later model version, with an unknown-value fallback.
* Keep `spatial_resolution` as the primary ALMA angular-resolution evidence
  and `s_resolution` as an independent cross-check.
* Keep line and continuum sensitivity fields semantically separate.
* Treat Archive sensitivity values as estimated candidate evidence rather
  than achieved product measurements.
* Preserve `qa2_passed` as evidence instead of applying it as a default
  retrieval filter.
* Preserve query, schema, and capture-time provenance because Archive rows
  and schemas are dynamic.
* Do not introduce a production dependency on FITS downloads, CASA
  processing, physical-product reconstruction, or undocumented Archive Web
  endpoints at this stage.

### Deferred follow-up requirements

The following work is documented but does not block current-cycle CSV
exploration:

1. Resolve the Archive Frequency Support Type representation gap.
2. Determine whether the nested mode value is available through a documented
   TAP table, stable API, DataLink, or another VO representation.
3. Preserve TAP field datatype, unit, UCD, arraysize, and description in a
   future Archive-ingestion metadata contract.
4. Revisit FDM/TDM-dependent duplication policy only after a stable,
   authoritative representation has been identified.
5. Validate policy-specific treatment of QA2 states with the project
   supervisor.

### Closure decision

Archive semantic exploration is complete for the current project scope.

The current evidence is sufficient to support:

* production Archive querying;
* query-completeness validation;
* metadata normalization;
* `obs_id` and `frequency_support` parsing;
* deterministic Source–Execution–SPW reconstruction;
* preservation of sparse and ambiguous associations;
* offline regression testing;
* live Archive smoke testing;
* transition to current-cycle CSV exploration.

This closure does not claim that every future Archive representation has
been exhausted. It establishes that the currently required structures are
understood, known values can be processed safely, and unsupported or unknown
semantics are reported explicitly rather than silently inferred.

### References

* [Cycle 13 ALMA Science Archive Manual](https://almascience.eso.org/documents-and-tools/cycle13/science-archive-manual)
* [ALMA query by spatial resolution](https://almascience.eso.org/alma-data/archive/archive-notebooks/nb5_ALMA_Query_by_spatial_resolution.html)
* [ALMA query by sensitivity](https://almascience.eso.org/alma-data/archive/archive-notebooks/nb7_ALMA_Query_by_sensitivity.html)
* [ALMA data resources](https://almascience.eso.org/alma-data)
* [ALMA processing resources](https://almascience.eso.org/processing)
